In [1]:
# Importa tudo

from selenium import webdriver
from selenium.webdriver.support.select import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

#Bibliotecas de Sistema
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

#pasta_downloads = r"C:\Users\dodonin\Downloads"
pasta_downloads = r"D:\Downloads"

navegador = eproc.novo_browser(pasta_downloads)

#configura variáveis
username = "dodonin"
password = keyring.get_password("eproc", username)
pyotop_code = "GJRGIYTCGBSGKYTEHE2TOZRUGFQTMMRQ"

#eproc.login_no_eproc_tj(navegador, username, password, pyotop_code)
eproc.login_no_eproc(navegador, username, password, pyotop_code)
# Entra no perfil da Vara
perfil = "SRD1CIV"
eproc.entrar_no_perfil(navegador, perfil)
eproc.entrar_nas_minutas(navegador)

Driver do Eproc importado
Perfil carregado: SRD1CIV


StaleElementReferenceException: Message: stale element reference: stale element not found
  (Session info: chrome=138.0.7204.96); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#stale-element-reference-exception
Stacktrace:
	GetHandleVerifier [0x0x7ff64b396f75+76917]
	GetHandleVerifier [0x0x7ff64b396fd0+77008]
	(No symbol) [0x0x7ff64b149dea]
	(No symbol) [0x0x7ff64b160224]
	(No symbol) [0x0x7ff64b15ecf3]
	(No symbol) [0x0x7ff64b1524e9]
	(No symbol) [0x0x7ff64b15039f]
	(No symbol) [0x0x7ff64b15482c]
	(No symbol) [0x0x7ff64b1548ff]
	(No symbol) [0x0x7ff64b1a245d]
	(No symbol) [0x0x7ff64b1932d3]
	(No symbol) [0x0x7ff64b1c846a]
	(No symbol) [0x0x7ff64b192c16]
	(No symbol) [0x0x7ff64b1c8680]
	(No symbol) [0x0x7ff64b1f065c]
	(No symbol) [0x0x7ff64b1c8243]
	(No symbol) [0x0x7ff64b191431]
	(No symbol) [0x0x7ff64b1921c3]
	GetHandleVerifier [0x0x7ff64b66d2ad+3051437]
	GetHandleVerifier [0x0x7ff64b667903+3028483]
	GetHandleVer

In [2]:
# Funções para manipulação de minutas
def trataMinuta(texto, tipo_ato):
    # Regex para capturar o texto a partir do tipo_ato até @NUMEROPROCESSOFORMATADO@
    # O re.escape é usado para escapar caracteres especiais no tipo_ato
    padrao = re.compile(
        rf"{re.escape(tipo_ato)}.*?(@NUMEROPROCESSOFORMATADO@)", 
        re.IGNORECASE | re.DOTALL | re.UNICODE
    )    
    match = padrao.search(texto)    
    if not match:
        return None  # Retorna None se não encontrou o padrão    
    texto_limpo = match.group(0)    
    # Remove tudo depois de @NUMEROPROCESSOFORMATADO@ (inclusive o que vier depois dele)
    texto_limpo = re.sub(r"(@NUMEROPROCESSOFORMATADO@).*", r"\1", texto_limpo, flags=re.DOTALL)    
    # Remove quebras de linha em excesso (mais de 2 quebras viram 2)
    texto_limpo = re.sub(r'\n\s*\n+', '\n\n', texto_limpo)    
    # Remove espaços em excesso nas linhas
    texto_limpo = '\n'.join(linha.strip() for linha in texto_limpo.splitlines())    
    return texto_limpo

def pegaMinuta(driver, cod_minuta):
    pyautogui.click(1000, 600)
    time.sleep(0.2)  # Pequena pausa para segurança
    driver.find_element(By.ID, "txtCodigoModelo").clear()
    time.sleep(0.2)  # Pequena pausa para segurança
    #insere o código 
    driver.find_element(By.ID, "txtCodigoModelo").click()
    time.sleep(0.2)  # Pequena pausa para segurança
    driver.find_element(By.ID, "txtCodigoModelo").send_keys(cod_minuta)
    time.sleep(2)
    pyautogui.hotkey('enter')
    time.sleep(4)
    # Localiza o elemento com código
    elemento = driver.find_element(By.PARTIAL_LINK_TEXT, str(cod_minuta))
    # Cria uma cadeia de ações e move o mouse até o elemento
    actions = ActionChains(driver)
    actions.move_to_element(elemento).perform()
    time.sleep(4)
    # Pega o texto
    pyautogui.click(1000, 600)
    time.sleep(0.5)  # Pequena pausa para garantir que o foco esteja correto
    pyautogui.hotkey('ctrl', 'a')
    time.sleep(0.2)  # Pequena pausa para segurança
    pyautogui.hotkey('ctrl', 'c')
    time.sleep(0.2)  # Dá tempo do sistema copiar para a área de transferência
    conteudo = pyperclip.paste()
    pyautogui.click(1000, 600)
    pyautogui.hotkey('f5')
    time.sleep(3)  # Pequena pausa para segurança
    #devolve o conteudo
    return conteudo

def gravaTextoMinuta(cod_minuta, texto_minuta):
    with closing(sqlite3.connect('minutas.db')) as conn:
        cursor = conn.cursor()
        query_insert = 'UPDATE minutas SET conteudo = ? WHERE Código = ?;'
        cursor.execute(query_insert, (texto_minuta, cod_minuta))
        conn.commit()

def pegaProximaMinutaVazia():
    with closing(sqlite3.connect('minutas.db')) as conn:
        cursor = conn.cursor()
        query_select = 'SELECT "Código", "Tipo de Documento" FROM minutas WHERE conteudo IS NULL LIMIT 1'
        cursor.execute(query_select)
        resultados = cursor.fetchall()
        return resultados

In [ ]:
#EXECUTA!
with closing(sqlite3.connect('minutas.db')) as conn:
    cursor = conn.cursor()
    query_select = 'SELECT COUNT(*) FROM minutas WHERE conteudo IS NULL'
    cursor.execute(query_select)
    resultado = cursor.fetchone()[0]  # pega o número da tupla (ex: (42,) -> 42)
    for _ in range(resultado):
        resultados = pegaProximaMinutaVazia()
        for r in resultados:
            try:
                cod_minuta = r[0]
                texto_minuta = pegaMinuta(navegador, cod_minuta)
                gravaTextoMinuta(cod_minuta, texto_minuta)  
                print(f"Texto da minuta {cod_minuta} capturado")  
            except:
                print("Erro. Passando para a próxima")


Texto da minuta 10000037873 capturado
Texto da minuta 10000054174 capturado
Texto da minuta 10000250772 capturado
Texto da minuta 10000054162 capturado
Texto da minuta 10000063927 capturado
Texto da minuta 10000095043 capturado
Texto da minuta 10000030712 capturado
Texto da minuta 10000030714 capturado
Texto da minuta 10000054159 capturado
Texto da minuta 10000247899 capturado
Texto da minuta 10000249287 capturado
Texto da minuta 10000244981 capturado
Texto da minuta 10000087686 capturado
Texto da minuta 10000238223 capturado
Texto da minuta 10000112024 capturado
Texto da minuta 10000247874 capturado
Texto da minuta 10000106588 capturado
Texto da minuta 10000184635 capturado
Texto da minuta 10000225452 capturado
Texto da minuta 10000195838 capturado
Texto da minuta 10000038369 capturado
Texto da minuta 10000064181 capturado
Texto da minuta 10000095812 capturado
Texto da minuta 10000064178 capturado
Texto da minuta 10000281538 capturado
Texto da minuta 10000059742 capturado
Texto da min

In [ ]:
conn.close()
